# A linearised instantaneous surface energy balance for the Arctic skin layer

`turbulent_flux_response.ipynb` answers Lynn's question — *how much of a cloud
longwave perturbation goes into turbulence?* — as a **covariance**: regress each
surface flux on the downwelling longwave across the record and read the slope.
That answer is correct, but it is not a partition of a perturbation, because
everything else that travels with an Arctic cloud (warm moist advection, wind,
stability) is inside the slope too.

This notebook answers the **perturbation** version instead. Perturb the
downwelling longwave, hold the atmosphere above fixed, let the skin temperature
find its new equilibrium, and ask which term takes up each W m⁻². That is a
well-posed question with a closed-form answer, and the answer is four numbers
that sum to one.

## Why the forcing may be treated as external

Arctic cloud radiative forcing is largely set by **synoptic advection** — the
clouds arrive from elsewhere — while the skin layer responds within minutes to
hours. On the timescale of the response the forcing is therefore external, and
that asymmetry is what licenses treating DLR as the independent variable. It is
the same licence Miller et al. (2017) and Sledd et al. (2025) take when they
regress SEB response terms on $(LW\downarrow + SW_{net})$. The
cloud–boundary-layer feedback Lynn describes is real, but it operates on the
boundary layer over longer timescales and through a much smaller loop gain than
the direct skin response.

Note what this framing does **not** claim. It does not claim that a particular
joule of longwave became a particular joule of turbulence. It measures how the
system responds to a perturbation, which is well posed regardless of the
circularity — and that sidesteps the attribution problem entirely.

## The model

The skin layer has no heat capacity, so its energy balance is *diagnostic*: the
fluxes into it must sum to zero at every instant. Linearising each flux about
the mean state in the skin temperature gives

$$\Delta \mathrm{DLR} \;=\; \big(\lambda_{LW} + \lambda_{SH} + \lambda_{LH} + \lambda_{G}\big)\, \Delta T_{skin} \tag{1}$$

and the fraction of the perturbation absorbed by each term is simply that
term's $\lambda$ divided by the sum. Every $\lambda$ is a **positive damping
coefficient** in W m⁻² K⁻¹:

| term | expression | mechanism |
|---|---|---|
| $\lambda_{LW}$ | $4\epsilon\sigma T^3$ | more emission as the skin warms |
| $\lambda_{SH}$ | $\rho c_p C_H U$ | stronger upward sensible heat flux |
| $\lambda_{LH}$ | $\rho L_s C_E U \,dq_{sat}/dT$ | stronger sublimation |
| $\lambda_{G}$ | see below | stronger conduction into the ice |

## This is not an analogy — it is ERA5's own surface equation

The IFS solves, for each surface tile $i$, **exactly this balance** with a
zero-heat-capacity skin (IFS Documentation Cy41r2, Part IV, eq. 8.22):

$$(1-f_{Rs,i})(1-\alpha_i)R_s + \epsilon\big(R_T - \sigma T_{sk,i}^4\big) + H_i + L_{v,s}E_i \;=\; \Lambda_{sk,i}\big(T_{sk,i} - T_1\big)$$

and the net longwave is then **re-linearised about the new skin temperature
using precisely $\epsilon\,4\sigma T_{sk}^3$** (their eq. 8.23). So equation (1)
is the IFS skin equation with the quartic linearised and the bulk fluxes
written out. The coefficients below are therefore not an external model imposed
on ERA5 output — they are ERA5's own coefficients, which means any disagreement
between (1) and the regressions in `turbulent_flux_response.ipynb` is
*informative* rather than a units error.

## Where each coefficient comes from

Nothing here is a literature value plugged in unexamined. Two of the four are
**measured from this record**, one is theory that the data can check, and one
is theory that the data cannot check but whose provenance is documented.

| $\lambda$ | source | value | citation |
|---|---|---|---|
| $\lambda_{LW}$ | **theory, checked against data** | $4\epsilon\sigma T^3 \approx 3.7$ W m⁻² K⁻¹ at 255 K | Stefan–Boltzmann; $\epsilon$ from IFS Cy41r2 §2.8.5 + Table 2.8 |
| $\lambda_{SH}$ | **measured, per surface class** | 3.7 (land) – 19.4 (MIZ), 9.9 for sea ice | this record; the direct SHF-on-$\Delta T$ regression |
| $\lambda_{LH}$ | **measured, cross-checked** | 0.3 – 12.1; 2.9 for sea ice | this record; vs Clausius–Clapeyron |
| $\lambda_{G}$ | **theory, frequency dependent** | 1.35 – 58 W m⁻² K⁻¹ | Carslaw & Jaeger (1959) §2.6; IFS ice properties eq. 8.149–8.150 |

### $\lambda_{LW}$ and the emissivity

The IFS builds its broadband surface emissivity by convolving a **two-band
spectral** emissivity with the Planck function at the skin temperature
(Cy41r2 §2.8.5):

- $\epsilon = 0.99$ everywhere **outside** the 800–1250 cm⁻¹ atmospheric window;
- inside the window, tile-dependent (Table 2.8): **0.99** open sea, **0.98** sea
  ice, **0.98** exposed snow, **0.93–0.96** vegetation and bare soil.

`broadband_emissivity` does that convolution rather than assuming a number. The
window holds only **23.4%** of the Planck emission at 250 K, so a window
emissivity 0.01 low pulls the broadband value down by 0.0023. **The emissivity
is not worth arguing about; the temperature is** — $\lambda_{LW}$ over midwinter
pack ice at 255 K is 3.7 W m⁻² K⁻¹ against 4.6 over open water at 273 K, a 24%
difference driven entirely by $T^3$.

### $\lambda_{SH}$, and the direction of the fit

$\lambda_{SH}$ is the coupling coefficient $\rho c_p C_H U$ of the bulk formula,
taken as $-d(\mathtt{msshf})/d(T_{skin}-T_{2m})$ per surface class. **The
direction of the regression matters and is not a presentation choice.** The
reciprocal of the *other* least-squares fit, $1/[d(\Delta T)/d(SHF)]$,
overstates the coupling by $1/r^2$ — a factor of 2.7 over pack ice, where
$r^2 = 0.37$. See `FIT_ORIENTS` in `turbulent_flux_response.py`; the module
reports both so the size of that error sits beside the number it would replace.

This is why the notes' quoted range of "roughly 9–26 W m⁻² K⁻¹" for sea ice
needs correcting: **9.9 is the coefficient and 26.7 is the $1/r^2$-inflated
inversion of the wrong fit.** The honest spread is *across surfaces* — 9.9 for
pack ice, 19.4 at the ice edge, 3.7 over land — not a range of uncertainty for
one surface. Both are carried through below so the consequence is visible.

### $\lambda_{LH}$

Measured the same way, as $-d(\mathtt{mslhf})/d(T_{skin}-T_{2m})$, and
cross-checked against the **saturated-skin** limit

$$\frac{\lambda_{LH}}{\lambda_{SH}} = \frac{L_s}{c_p}\,\frac{dq_{sat}}{dT} \approx 0.199 \ \text{at 255 K}$$

which is the right limit over snow and ice, because a frozen skin is saturated
by construction and there is no moisture-availability factor to guess. For sea
ice that predicts 2.0 W m⁻² K⁻¹ against a measured 2.9 — the same size, which
is the evidence that regressing the latent heat flux on a *temperature*
difference is a legitimate reduction. **The notes set $\lambda_{LH}$ to zero;
it is not zero, and over the ice edge it is 9.5 W m⁻² K⁻¹.**

## The conduction term is frequency dependent — by a factor of forty

This is the part of the model that needs the most care, and the part the
original notes got backwards.

A slab of ice does not present one conductance to the surface. Force the surface
sinusoidally at angular frequency $\omega$ and the heat only penetrates a
**diffusive damping depth**

$$d(P) = \sqrt{\frac{\kappa P}{\pi}}, \qquad \kappa = \frac{k}{\rho c}$$

so the conductance the skin feels is roughly $k/d(P)$ — large for a fast
perturbation, small for a slow one. Exactly, for a slab of thickness $h$ over a
base held at the freezing point (Carslaw & Jaeger 1959, §2.6),

$$Y(\omega) = k\,q\,\coth(q h), \qquad q = \sqrt{\frac{i\omega\rho c}{k}} \tag{2}$$

whose **real part** is the component in phase with the surface temperature
perturbation, and that is the damping coefficient that belongs in (1). Both
limits fall out of (2) correctly: $Y \to k/h$ as $\omega \to 0$, and
$Y \to k\sqrt{i\omega\rho c/k}$ (the semi-infinite result) at high frequency.

Evaluated on **ERA5's own ice column** — $k = 2.03$ W m⁻¹ K⁻¹,
$\rho c = 1.88\times10^6$ J m⁻³ K⁻¹, $h = 1.5$ m in four layers, base at
$T_0 - 1.7$ K, *no snow* (IFS Cy41r2 eq. 8.149–8.150 and §8.9 assumption iii):

| forcing period | $\lambda_G$ [W m⁻² K⁻¹] | damping depth |
|---|---|---|
| 1 hour | **57.7** | 3.5 cm |
| 6 hours | 23.6 | 8.6 cm |
| 12 hours | 16.7 | 12.2 cm |
| 1 day | 11.8 | 17.2 cm |
| 1 week | 4.5 | 45.6 cm |
| 1 month | 2.0 | 94.4 cm |
| steady state | **1.35** | whole slab |

### Two things to notice

**First, the one-hour value is not arbitrary.** The IFS gives its top ice layer
a depth $D_1 = 0.07$ m, so $2k/D_1 = 58.0$ W m⁻² K⁻¹ — and 3.5 cm is *exactly*
the hourly damping depth in ice. The IFS layer discretisation is built so that
its top-layer conductance **is** the hourly diffusive admittance, and the skin
conductivity it uses for the ice-cap tile is 58.0 W m⁻² K⁻¹ (Table 2.8, "Ice
caps and glaciers"). The figure below puts that value on the curve so the
agreement can be seen rather than asserted.

**Second — and this corrects the note this notebook was written from — the
conduction term does not grow for a sustained forcing, it shrinks**, from 58 to
1.35 W m⁻² K⁻¹. A slower perturbation has to push heat further into the ice,
which puts more thermal resistance *in series* with the skin. The conductive
**flux** shrinks too: 21 W m⁻² at one hour against 2.3 W m⁻² in steady state for
a −27 W m⁻² perturbation, because the fall in $\lambda_G$ outruns the rise in
$\Delta T_{skin}$. What genuinely does grow with time is the **ice-growth**
response, which is a different question, is not a term in (1) at all, and needs
a time-dependent column model.

The figure below needs no ERA5 data — only the column properties — so it can be
drawn before anything is loaded.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import turbulent_flux_response as tfr
import linearized_seb_model as lsm

# open_mfdataset warns about join/compat defaults changing; the modules pin
# both explicitly, so the warning is noise here.
warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams["figure.dpi"] = 110      # on-screen only; saved files use dpi=

# Set to a directory to write PNGs as well as display them; None displays only.
SAVE_DIR = Path("figures/linearized_seb")
DPI = 350

# The forcing period the conduction term is evaluated at. THE SINGLE MOST
# CONSEQUENTIAL SETTING IN THIS NOTEBOOK -- lambda_G spans a factor of 40 across
# the plausible range, so figure 2 sweeps it rather than asking you to trust one
# value. Twelve hours is the middle of the synoptic band.
PERIOD_S = 12 * 3600.0

# The perturbation the response is quoted for: the cloud longwave forcing this
# project has been measuring, taken negative so the numbers read as a cooling.
# The FRACTIONS do not depend on it; only dT_skin scales.
D_DLR = -27.0

In [ ]:
# lambda_G against forcing period, for both columns, with the two limits and
# the IFS's own top-layer conductance marked. No data needed.
lsm.fig_ground_admittance();

## The air-coupling factor, and why the model and the regression must disagree

Equation (1) holds $T_{2m}$ fixed. In the record it is not fixed: an hour with
more DLR is an hour with a warmer air mass, and the turbulent fluxes respond to
the **difference**. Writing $\Delta T_{2m} = \alpha\,\Delta T_{skin}$, the
balance becomes

$$\Delta \mathrm{DLR} = \big[\lambda_{LW} + (1-\alpha)(\lambda_{SH} + \lambda_{LH}) + \lambda_{G}\big]\,\Delta T_{skin} \tag{3}$$

so the turbulent damping is scaled by $(1-\alpha)$. Measured on this record,

$$\alpha = \frac{dT_{2m}/d\mathrm{LWD}}{dT_{skin}/d\mathrm{LWD}}$$

is **1.00 over sea ice**, 0.99 over land, 1.05 at the ice edge — the air warms
right along with the surface. That collapses the turbulent term to nothing, and
it is precisely why `turbulent_flux_response.ipynb` finds $|f_{SH}| \le 0.14$
where (1) with $\alpha = 0$ predicts 0.30–0.55.

### Both are correct answers to different questions

| limit | the question it answers | bound |
|---|---|---|
| $\alpha = 0$ | *If a cloud added 27 W m⁻² to this surface and nothing else changed, where would it go?* The model-physics question — what a single-column or offline-surface experiment would answer. | **upper** bound on the turbulent share |
| $\alpha$ measured | *Across the hours in this record, how did the fluxes actually co-vary with DLR?* What the reanalysis shows, and what an observational comparison can check. | **lower** bound on the turbulent share |

Reporting only the first overstates turbulence by roughly a factor of five;
reporting only the second understates the surface's own physics. Every table and
figure below draws the bracket, and `partition(lam, alpha=...)` takes either.

**$\alpha > 1$ is not a rounding artefact.** It means the air overshoots the
surface, so the turbulent term becomes an *amplifier* rather than a damper; and
once $(\alpha-1)(\lambda_{SH}+\lambda_{LH})$ exceeds $\lambda_{LW}+\lambda_G$
the sum in (3) goes negative and the linearised balance has **no stable
solution at all**. That is exactly what the open-water row reports
($\alpha = 6.88$), and it is a real statement about ERA5, not a bug: ERA5
*prescribes* the sea surface temperature from the OSTIA analysis (IFS Cy41r2
§8.10), so the water skin is clamped, the whole skin-to-air difference is the
air moving, and neither (1) nor (3) applies there. Those rows are marked rather
than plotted.

## Load the data

The slow cell, and the only one that touches the archive. This is the **same
`prepare()` call** as `turbulent_flux_response.ipynb`, with the same population
and the same fractional phase scheme, so the two notebooks describe exactly the
same cell-hours and the $\lambda_{SH}$ used here is the coefficient that
notebook's figure 3 draws.

- **cloudy**: `tcc >= 0.99` (effectively overcast)
- **liquid-bearing**: the union of the *liquid-only* and *mixed-phase*
  categories of `plot_lwp_histogram_by_surface_class.py`, under the fractional
  scheme ($LWP/CWP \geq 0.90$ liquid-only, $IWP/CWP \geq 0.85$ ice-only, mixed
  the rest, with a 0.01 g m⁻² floor on each species)

14.08 million cell-hours over the Barrow strip, Oct–Mar, seasons 2022/23–2025/26.

In [ ]:
A = tfr.prepare(
    region="barrow",
    years=tuple(range(2022, 2026)),   # season START years -> 2022/23 .. 2025/26
    season_start=(10, 1),
    season_end=(3, 31),

    # --- cloud phase: fractional scheme, identical to
    # --- turbulent_flux_response.ipynb and comparison_with_Genie_Obs.ipynb
    phase_mode="fraction",
    liquid_fraction_min=0.90,         # LWP/CWP at or above this = liquid only
    ice_fraction_min=0.850,           # IWP/CWP at or above this = ice only
    min_lwp=0.01,                     # g m-2, liquid below this counts as absent
    min_iwp=0.01,                     # g m-2, ice below this counts as absent

    min_cloud_fraction=0.99,          # overcast gate
    lsm_tol=0.01,                     # land-sea mask tolerance
    open_ocean_max_siconc=0.05,       # below this: open ocean
    sea_ice_min_siconc=0.95,          # above this: pack ice; between: MIZ
    storage="local",
    dpi=DPI,
)

## Numeric report

Every number the figures draw. Five blocks:

1. **The four coefficients**, with $\lambda_{LW}$ theory beside the measured
   $d(LWU)/dT_{skin}$ — *the one independent check in the table* — and
   $\lambda_{LH}$ beside its Clausius–Clapeyron estimate.
2. **The partition at $\alpha = 0$**, the fixed-atmosphere limit.
3. **The partition at the measured $\alpha$**, the realised-covariance limit.
4. **Closure against the regressions**, including the inverse problem: given
   ERA5's own total damping and its measured $\alpha$, what $\lambda_G$ would the
   balance need, and what forcing timescale implies it?
5. **The timescale sweep** for sea ice, and the published response shares from
   Miller et al. (2017) and Sledd et al. (2025) for comparison.

In [ ]:
lsm.print_report(A, period_s=PERIOD_S, d_dlr_W_m2=D_DLR)

## 1. The four coefficients, by surface class

Panel **(a)** stacks the four $\lambda$, so the bar height is the model's total
damping — the reciprocal of the model's $dT_{skin}/d\mathrm{LWD}$ — and the
segments are the partition. The open circles are ERA5's **own** total damping,
$1/[dT_{skin}/d\mathrm{LWD}]$ from the regression; they are drawn as markers
rather than a fifth bar because they are a different estimator and should not
look like another term.

Panel **(b)** is the same information as fractions, which is the direct answer to
"how much goes into turbulence". Left bar of each pair is $\alpha = 0$, right
(hatched) is the measured $\alpha$ printed under the class name.

Note the conducting column each class was given, in the box on (a): 1.5 m of
ERA5 sea ice under the ice classes, 2.89 m of frozen soil under land and
coastal. Applying one column to every class would be simpler and would also be
wrong.

In [ ]:
lsm.fig_lambda_bars(A, out_dir=SAVE_DIR, dpi=DPI, period_s=PERIOD_S);

## 2. How much goes into turbulence? It depends on how long the cloud stays

**The figure this notebook is for.** $\lambda_{LW}$, $\lambda_{SH}$ and
$\lambda_{LH}$ do not depend on the forcing timescale, but $\lambda_G$ spans a
factor of forty across it — so the answer to Lynn's question is a **curve, not
a number**. The shaded band is 6–48 hours, where Arctic cloud radiative forcing
actually lives.

Top panels: the four fractions, stacked. Bottom panels: $\lambda_G$ and
$\sum\lambda$ on the left axis, $|\Delta T_{skin}|$ for a −27 W m⁻²
perturbation on the right (red). $\lambda_{SH}$ is drawn as a horizontal dotted
line to make the crossing explicit — **conduction dominates faster than the
crossing period and turbulence dominates slower than it**. For ERA5 pack ice
that crossing sits at a **34-hour** period, just past the top of the synoptic
band; at the ice edge, where $\lambda_{SH} = 19.4$, it sits at **8.9 hours**,
inside it. So the ice edge is turbulence-dominated for a typical cloud episode
and the pack is conduction-dominated — a real difference between surfaces, and
one the covariance regressions cannot see.

In [ ]:
lsm.fig_partition_vs_timescale(A, out_dir=SAVE_DIR, dpi=DPI);

## 3. The answer with a number on it

Panel **(a)**: the skin temperature change a −27 W m⁻² perturbation produces,
per class, at three forcing timescales, with ERA5's own regression response
marked as open circles. The circles cool *further* than the fixed-atmosphere
bars, which is the $\alpha$ effect again: with the air free to cool alongside
the surface, less damping is available and the skin moves more.

Open ocean is the exception, and for the opposite reason — ERA5 prescribes its
surface temperature, so its skin barely moves at all (−0.46 K).

Panel **(b)**: the same partition expressed as flux changes in W m⁻², which is
the physically legible form, because the four bars sum to the perturbation
itself (the dashed line).

In [ ]:
lsm.fig_forcing_response(A, out_dir=SAVE_DIR, dpi=DPI, d_dlr_W_m2=D_DLR);

## 4. Closure check — does the model reproduce ERA5's own response?

Three tests, in increasing difficulty:

**(a) $\lambda_{LW}$. This one must close.** Theory $4\epsilon\sigma T^3$ against
the measured $d(LWU)/dT_{skin}$. If these disagreed, something would be wrong
with the emissivity or the units, not with the physics. They agree to **+0.3%
to +2.3%** across every class — which validates the Planck-weighted emissivity,
the linearisation, and the moment machinery in one stroke.

A caution on reading this panel: $f_{LWU} = \lambda_{LW}\times dT_{skin}/d\mathrm{LWD}$
is an **identity**, not evidence, so the fact that the model's $f_{LW}$ at
$\alpha \approx 1$ reproduces the regression's $f_{lwu}$ to three decimals is
arithmetic. Panel (a) is the part that is a real test.

**(b) The total damping.** The model's $\sum\lambda$ against ERA5's
$1/[dT_{skin}/d\mathrm{LWD}]$, with the ratio printed. These are *not* expected
to agree, and the ratio is the size of the air-coupling effect: 4.3× over sea
ice, 7.4× at the ice edge, and 0.9× over open water, where ERA5's clamped SST
makes it the model that under-damps.

**(c) The turbulent share, bracketed.** The model's $f_{SH}$ at both $\alpha$
limits (the blue bar) against ERA5's regression $f_{SH}$ (open circle) and
Miller et al.'s 11% at Summit (dashed green). **The bracket does not quite
contain the regression value** — over sea ice it runs $[-0.002, +0.298]$ against
a measured $-0.049$ — and that gap is the honest residual of the exercise. It is
small in absolute terms and it has an identifiable cause: at $\alpha = 1.0033$
the turbulent terms are being driven by the *sign* of a third-decimal-place
number, and the rest of ERA5's $f_{SH}$ comes from wind and stability travelling
with the cloud, which equation (3) has no term for.

In [ ]:
lsm.fig_model_vs_era5(A, out_dir=SAVE_DIR, dpi=DPI, period_s=PERIOD_S);

## Reproducing the original notes exactly

Worth doing explicitly, because the notes' headline numbers — *66–84% into
turbulence, 11–26% into upwelling longwave, 4–9% into conduction, and 0.9–1.9 K
of skin cooling for −27 W m⁻²* — are recovered **exactly** by this
implementation under three specific assumptions:

1. $\lambda_{LH} = 0$;
2. $\lambda_G$ at its **steady-state** value (1.35 bare, 0.59 with 30 cm snow);
3. $\lambda_{SH}$ swept from 9.90 to 26.72.

The cell below runs those cases. It matters because it separates *"the
arithmetic is wrong"* from *"the assumptions are debatable"* — the arithmetic
is right, and the next section is about the assumptions.

In [ ]:
# The notes' own configuration: no latent term, steady-state conduction, and
# lambda_SH swept over the range the notes quote.
lam_ice = lsm.lambdas(A, "sea_ice", period_s=PERIOD_S, column="sea_ice")

print(f"{'lam_SH':>7}  {'column':<20}{'sum':>7}{'f_SH':>8}{'f_LW':>8}"
      f"{'f_G':>7}{'dT_skin':>9}")
for lam_sh in (lam_ice.sh, -1.0 / tfr.fit_pair(
        A.acc(), tfr.CLASS_CODES["sea_ice"], "shf_W_m2",
        "dskt_t2m_K")["y_on_x"]["slope"]):
    for col in ("sea_ice", "sea_ice_snow"):
        L = lam_ice._replace(sh=lam_sh, lh=0.0, g=lsm.lambda_g_steady(col))
        p = lsm.partition(L, D_DLR, alpha=0.0)
        print(f"{lam_sh:7.2f}  {col:<20}{p['lambda_sum']:7.2f}"
              f"{100 * p['f_sh']:7.1f}%{100 * p['f_lw']:7.1f}%"
              f"{100 * p['f_g']:6.1f}%{p['dT_skin_K']:9.2f} K")
print("\nthe second lam_SH is the 1/r^2-inflated inversion of the wrong fit --")
print("reported here only because the notes quote it as the top of a range.")

In [ ]:
# And the same surface with lambda_LH measured rather than assumed zero, at
# both ends of the timescale range. This is the substantive change.
print(f"{'lambda_G':>10}  {'sum':>7}{'f_LW':>8}{'f_SH':>8}{'f_LH':>8}"
      f"{'f_G':>7}{'dT_skin':>9}")
for g, tag in ((lsm.lambda_g(PERIOD_S, "sea_ice"), "12 h"),
               (lsm.lambda_g_steady("sea_ice"), "steady")):
    p = lsm.partition(lam_ice._replace(g=g), D_DLR, alpha=0.0)
    print(f"{tag:>10}  {p['lambda_sum']:7.2f}{100 * p['f_lw']:7.1f}%"
          f"{100 * p['f_sh']:7.1f}%{100 * p['f_lh']:7.1f}%"
          f"{100 * p['f_g']:6.1f}%{p['dT_skin_K']:9.2f} K")

## What this model actually says

Barrow strip, Oct–Mar, seasons 2022/23–2025/26, overcast and liquid-bearing
under the fractional phase scheme: 14.08 million cell-hours, of which 8.30
million are pack ice. $\alpha = 0$ unless noted.

### 1. The coefficients, measured

| class | $T_{skin}$ | $\lambda_{LW}$ | $\lambda_{SH}$ | $r^2$ | $\lambda_{LH}$ | $\lambda_{LH}$ (C–C) | $\alpha$ |
|---|---|---|---|---|---|---|---|
| Land | 260.2 | 3.91 | 3.66 | 0.13 | 0.30 | 1.14 | 0.99 |
| Coastal | 261.6 | 3.99 | 14.45 | 0.66 | 7.23 | 5.06 | 1.01 |
| Open ocean | 273.2 | 4.58 | 19.15 | 0.80 | 12.10 | 16.79 | 6.88 |
| Marginal ice zone | 263.4 | 4.10 | 19.35 | 0.83 | 9.54 | 7.84 | 1.05 |
| **Sea ice** | **255.0** | **3.71** | **9.90** | 0.37 | **2.93** | 1.97 | **1.00** |

(W m⁻² K⁻¹.) Two things worth noting. The measured $\lambda_{LH}$ agrees with
the saturated-skin Clausius–Clapeyron estimate within a factor of 1.5 for every
class, which is the evidence that regressing a moisture flux on a temperature
difference is a legitimate reduction over a frozen surface. And **$\alpha$ is
within 5% of one for every class except open water** — the near-surface air in
ERA5 is tightly bolted to the surface beneath it.

### 2. $\lambda_{LW}$ closes to better than 2.4%

The one genuinely independent check available. Theory $4\epsilon\sigma T^3$ with
the Planck-weighted IFS emissivity, against the measured $d(LWU)/dT_{skin}$:
+0.8% (land), +0.9% (coastal), +0.6% (ocean), +0.3% (MIZ), +2.3% (sea ice). The
linearisation, the emissivity convolution and the moment machinery are all
consistent.

### 3. The answer to Lynn's question is a curve, and it crosses inside the synoptic band

For ERA5 pack ice at $\alpha = 0$:

| forcing period | $f_{LW}$ | $f_{SH}$ | $f_{LH}$ | $f_G$ | $\Delta T_{skin}$ (−27 W m⁻²) |
|---|---|---|---|---|---|
| 1 hour | 5.0% | 13.3% | 3.9% | **77.7%** | −0.36 K |
| 12 hours | 11.2% | 29.8% | 8.8% | **50.2%** | −0.81 K |
| 1 day | 13.1% | 35.0% | 10.3% | 41.6% | −0.95 K |
| 1 week | 17.7% | 47.1% | 13.9% | 21.3% | −1.29 K |
| steady state | 20.7% | **55.3%** | 16.3% | 7.6% | −1.51 K |

Turbulence (SH + LH) takes **39%** on a synoptic timescale and **72%** for a
sustained forcing. $\lambda_G$ crosses $\lambda_{SH}$ at a **34-hour** period
over pack ice and a **8.9-hour** period at the ice edge — so a typical cloud
episode is conduction-dominated over the pack and turbulence-dominated at the
edge.

### 4. So the fixed-atmosphere model over-damps ERA5 by a factor of 4 to 8

| class | model $\sum\lambda$ | ERA5 $1/[dT_{skin}/d\mathrm{LWD}]$ | ratio |
|---|---|---|---|
| Land | 25.8 | 5.06 | 5.1× |
| Coastal | 43.6 | 5.41 | 8.1× |
| Open ocean | 52.5 | 58.7 | 0.9× |
| Marginal ice zone | 49.7 | 6.69 | 7.4× |
| Sea ice | 33.2 | 7.65 | 4.3× |

That is not an inconsistency, it is equation (3) with $\alpha \approx 1$. Put the
measured $\alpha$ back in and the turbulent terms vanish; what is left is
$\lambda_{LW} + \lambda_G$, and requiring **that** to equal ERA5's own total
damping is a subtraction, not a fit:

| class | $\lambda_G$ needed | implies a forcing period of |
|---|---|---|
| Marginal ice zone | 3.98 | 8.8 days |
| **Sea ice** | **3.98** | **8.8 days** |

The two ice classes land on the same implied timescale from independent
regressions, which is suggestive rather than conclusive — treat it as a
diagnostic, since ERA5's residual term also carries storage, melt and any
imbalance, and the regression mixes a whole spectrum of forcing durations. But
it says the record behaves as though the ice column were responding on a
roughly week-to-10-day timescale, not an hourly one.

### 5. Damping over sea ice is larger, not smaller

Worth stating plainly because it contradicts the §4.3 conclusion the notes were
sceptical of. Pack ice has the *weakest* radiative damping of any class
($\lambda_{LW} = 3.71$ against 4.58 over water, a 19% difference from $T^3$
alone) and the *weakest* turbulent coupling ($\lambda_{SH} = 9.90$ against 19.4
at the ice edge — a stable boundary layer over a smooth cold surface). Both
effects point the same way and would make the ice surface *more* volatile. What
makes the total damping over pack ice larger than over the ice edge in the
regression (7.65 against 6.69) is neither of those: it is that ERA5's ice column
absorbs more, which is the $\lambda_G$ term.

## Three corrections to the original notes

Stated explicitly because two of them change the answer.

**1. The conduction term shrinks for a sustained forcing, it does not grow.**
The note says "for a sustained forcing the ice column itself cools and the
conduction term grows". It falls, from 58 to 1.35 W m⁻² K⁻¹, because a slower
perturbation must push heat deeper and therefore meets more thermal resistance
in series with the skin. The conductive *flux* falls too — 21.0 W m⁻² at one
hour against 2.0 W m⁻² in steady state. What does grow with time is the
**ice-growth** response, which is a genuinely time-dependent problem, is not a
term in equation (1), and needs a column model rather than a skin balance. The
§5 ice-growth question is therefore untouched by any of this.

**2. $\lambda_{SH} \in [9, 26]$ for sea ice is not an uncertainty range.** 9.9
is the coupling coefficient from the correct regression; **26.7 is the
$1/r^2$-inflated reciprocal of the wrong fit** — the error
`turbulent_flux_response.py` was written to prevent, and it is a factor of 2.7
at the sea-ice $r^2$ of 0.37. Quoting the pair as a range makes the turbulent
share look like 66–84% when the correct coefficient alone gives 55% at steady
state and 30% at a 12-hour timescale. The real spread in $\lambda_{SH}$ is
**across surfaces** — 3.7 land, 9.9 pack ice, 19.4 ice edge — which is the
physically interesting variation and is what the figures show.

**3. $\lambda_{LH}$ is not negligible.** Setting it to zero is what produces the
66% turbulent share; the measured value of 2.93 W m⁻² K⁻¹ over pack ice moves
the sensible-heat share from 66% to 55% at steady state. Over the ice edge
$\lambda_{LH} = 9.54$, half of $\lambda_{SH}$, and dropping it would be a large
error. The saturated-skin Clausius–Clapeyron limit
$\lambda_{LH}/\lambda_{SH} = (L_s/c_p)\,dq_{sat}/dT \approx 0.199$ at 255 K says
this is expected, not surprising.

**One thing the notes got exactly right**, and it is the important one: the
perturbation-and-response framing rather than energy accounting. That is the
standard framing, it is what Miller et al. (2017) and Sledd et al. (2025) do,
and it sidesteps the attribution problem cleanly.

## Comparison with the two published studies

| study | surface | method | LW↑ | SH | LH | conduction |
|---|---|---|---|---|---|---|
| Miller et al. (2017), annual | Summit ice sheet | regression on $LW\downarrow + SW_{net}$ | 77% | 11% | 1.5% | 10% (+6% storage) |
| Miller et al. (2017), winter | Summit ice sheet | same | 65–85% | ~11% | <1% | 23% |
| This model, $\alpha$ measured | ERA5 pack ice | linearised skin balance | 18% | ~0% | ~0% | 82% |
| This model, $\alpha = 0$, 12 h | ERA5 pack ice | linearised skin balance | 11% | 30% | 9% | 50% |
| ERA5 regression | Barrow pack ice | regression on $LWD$ | 50% | −5% | −1% | 54% (residual) |

**Miller's regressions are the same estimator as the ERA5 regression column**,
so those are the two rows to compare. They disagree sharply: 77% of the response
into upwelling longwave at Summit against 50% over Barrow pack ice, and 11% into
sensible heat against −5%. Two reasons, and they are both real:

- **Summit is a 3 km-high ice sheet with a very deep, cold, dry snowpack.**
  Miller's measured snow conductivity is 0.47 W m⁻¹ K⁻¹ (density 413 kg m⁻³)
  against 0.31 for Arctic sea-ice snow (Sturm et al. 1997), and the conductive
  flux is computed at 20–40 cm depth. A surface that cannot dump heat downward
  must radiate it, which is exactly the 77%.
- **ERA5's sea ice has no snow at all** (IFS Cy41r2 §8.9, assumption iii), so
  its 1.5 m bare ice slab is a far better conductor than the real surface. That
  is why the residual/conduction share over Barrow pack ice is 54% where
  Miller's is 10%, and it is a model artefact rather than a physical difference.
  `COLUMNS["sea_ice_snow"]` sizes it: 30 cm of snow in series drops the
  steady-state $\lambda_G$ from 1.35 to 0.59, and the 12-hour value from 16.7 to
  3.4 W m⁻² K⁻¹ — a factor of five.

**Sledd et al. (2025)** report that during winter ice growth the forcing change
appears in upwelling longwave, sensible heat *and* subsurface heat flux, and
that in summer melt the surface temperature is fixed so the response goes into
melt instead. The winter three-way split is qualitatively what equation (1)
gives; the summer regime is the $\lambda_G \to \infty$ clamp, the same limit
ERA5's prescribed SST imposes on the open-ocean class here.

## Caveats — read these before quoting any number

**1. This is the instantaneous skin balance, made timescale-aware only through
$\lambda_G$.** The heat capacity of the ice column is inside equation (2), but
nothing here evolves. For a forcing sustained over weeks the ice column cools,
its temperature profile changes, and the ice-growth response begins — a
time-dependent problem that a zero-heat-capacity skin balance cannot address.
The §5 ice-growth question needs a column model.

**2. $\lambda_{SH}$ inherits whatever bias ERA5's skin temperature has, and over
Arctic sea ice that bias is known and substantial.** The IFS carries **no snow
layer** on sea ice, which warms the modelled ice surface by 5–10 K in winter
(Batrak & Müller 2019). A skin temperature biased warm sits on the wrong side of
the stability transition, so $\lambda_{SH} = 9.90$ is the coupling coefficient of
a boundary layer that is *less stable* than the real one — an upper bound on the
real coupling. This is the same concern §4.3 raises, and it applies to the
$\lambda_G$ half of the model too, which is why `COLUMNS["sea_ice_snow"]` exists.

**3. $\lambda_{SH}$ over land and at the ARM site is unmeasured, not small.**
$r^2 = 0.13$ and $0.18$ respectively. Read those rows as "no constraint", not
as "weak coupling".

**4. Every $\lambda_{SH}$ and $\lambda_{LH}$ is a covariance across synoptic
variability**, not a controlled perturbation, so each carries the same
confounding the parent notebook documents at length. What the linearised model
adds is that it *separates* the coefficient from the air-mass response, and puts
the size of that response — $\alpha$ — on the page as a number.

**5. The forcing period is an assumption, and it is the most consequential one.**
$\lambda_G$ spans a factor of forty across the plausible range. Nothing in this
notebook measures the timescale of Arctic cloud forcing; the 12-hour default is
a judgement about the synoptic band, and figure 2 exists so the reader can
substitute their own.

### How to state this

**As a framework with numbers attached, not as a finished result.** The
framework is defensible — it is ERA5's own skin equation — and two of its four
coefficients are measured from 8.3 million cell-hours of pack ice. But the
answer to "how much goes into turbulence" is *39% on a synoptic timescale, 72%
for a sustained forcing, and near zero in the realised covariance because the
air moves with the surface* — three numbers, with the reasons for the spread
identified. That is still far more than the project had before.

## Sources

**Papers**

- Batrak, Y. and Müller, M. (2019). On the warm bias in atmospheric reanalyses
  induced by the missing snow over Arctic sea-ice. *Nature Communications* **10**,
  4170. https://doi.org/10.1038/s41467-019-11975-3
- Carslaw, H. S. and Jaeger, J. C. (1959). *Conduction of Heat in Solids*, 2nd
  ed., Oxford. §2.6 — the periodic-forcing damping depth and the surface
  admittance of a slab, equation (2) here.
- Maykut, G. A. (1978). Energy exchange over young sea ice in the central Arctic.
  *J. Geophys. Res.* **83**, 3646–3658. — conductive fluxes through thin ice.
- Miller, N. B., Shupe, M. D., Cox, C. J., Noone, D., Persson, P. O. G. and
  Steffen, K. (2017). Surface energy budget responses to radiative forcing at
  Summit, Greenland. *The Cryosphere* **11**, 497–516.
  https://doi.org/10.5194/tc-11-497-2017
- Sledd, A., Shupe, M. D., Solomon, A. and Cox, C. J. (2025). Surface Energy
  Balance Responses to Radiative Forcing in the Central Arctic From MOSAiC and
  Models. *J. Geophys. Res. Atmospheres* **130**, e2024JD042578.
  https://doi.org/10.1029/2024JD042578
- Stephens, G. L. (1978). Radiation profiles in extended water clouds. II:
  Parameterization schemes. *J. Atmos. Sci.* **35**, 2123–2132. — the cloud
  emissivity relation behind the LWP regimes.
- Sturm, M., Holmgren, J., König, M. and Morris, K. (1997). The thermal
  conductivity of seasonal snow. *J. Glaciology* **43**, 26–41.

**Model documentation** — every ERA5 parameter value in this notebook comes from
one place, [IFS Documentation Cy41r2, Part IV: Physical
Processes](https://www.ecmwf.int/en/elibrary/79697-ifs-documentation-cy41r2-part-iv-physical-processes),
which is what the ERA5 data documentation itself points to:

| what | where | value |
|---|---|---|
| surface emissivity, out of window | §2.8.5 | 0.99, all tiles |
| surface emissivity, 800–1250 cm⁻¹ window | Table 2.8 | 0.99 open sea, 0.98 sea ice, 0.98 exposed snow, 0.93–0.96 vegetation/soil |
| tiled skin energy balance | eq. 8.22 | zero-heat-capacity skin, $\Lambda_{sk}(T_{sk}-T_1)$ |
| longwave re-linearisation | eq. 8.23 | $\epsilon\,4\sigma T_{sk}^3$ |
| skin conductivity, ice caps/glaciers | Table 8.2 | 58.0 W m⁻² K⁻¹ ( = $2k/D_1$) |
| sea-ice heat diffusion | eq. 8.149 | $\lambda_I = 2.03$ W m⁻¹ K⁻¹, $(\rho C)_I = 1.88\times10^6$ J m⁻³ K⁻¹ |
| sea-ice slab depth and layers | eq. 8.150 | $D_I = 1.5$ m in four layers (0.07 / 0.21 / 0.72 / 0.50 m) |
| no snow on sea ice | §8.9, assumption (iii) | stated simplification |
| prescribed SST | §8.10 | OSTIA analysis, held fixed |
| soil layer depths | Table 8.6 | 0.07 / 0.21 / 0.72 / 1.89 m |
| moist thermodynamic constants | Ch. 12; eq. 7.5 | $L_s$, $c_p$, $R_v$; Tetens over ice |

Physical constants are CODATA 2018. The frozen-soil conductivity
(2.2 W m⁻¹ K⁻¹) and heat capacity ($2.0\times10^6$ J m⁻³ K⁻¹) are mid-range
values for moist frozen tundra rather than IFS-documented numbers, so the land
and coastal rows should be read as indicative — and their $\lambda_{SH}$ has
$r^2 = 0.13$ anyway.

**Sibling notebooks in this directory**

- `turbulent_flux_response.ipynb` — the covariance answer, and the source of
  $\lambda_{SH}$ and $\lambda_{LH}$
- `comparison_with_Genie_Obs.ipynb` — the phase scheme used here
- `plot_hovmoller_dlwd_dlat_by_season.ipynb` — the forcing gradient

## Changing options

Anything that alters what is **loaded, classified or accumulated** — region,
years, season window, phase thresholds, cloud-cover gate — needs a fresh
`tfr.prepare()`. Everything this module adds is computed from the accumulated
moments, so it costs nothing to change:

```python
# a different forcing timescale, or the steady-state limit
lsm.print_report(A, period_s=6 * 3600.0)
lsm.print_report(A, period_s=365 * 86400.0)

# put the snow back on the ice: the missing-snow error, sized
lsm.print_report(A, column="sea_ice_snow")
lsm.fig_lambda_bars(A, column="sea_ice_snow")

# the Clausius-Clapeyron latent term instead of the measured one
lsm.print_report(A, lh_source="clausius")

# all-sky hours instead of liquid-bearing overcast
lsm.print_report(A, population="all")

# a different perturbation amplitude (the FRACTIONS do not change)
lsm.print_report(A, d_dlr_W_m2=+10.0)
```

Individual numbers, without a figure:

```python
lam = lsm.lambdas(A, "sea_ice", period_s=12 * 3600.0)
lam.lw, lam.sh, lam.lh, lam.g          # the four coefficients
sum(lam[:4])                            # the total damping

p = lsm.partition(lam, d_dlr_W_m2=-27.0)
p["f_sh"], p["dT_skin_K"], p["dflux_sh_W_m2"]
p["stable"]                             # CHECK THIS before quoting anything

# the air-coupling factor, and the partition it implies
a = lsm.air_coupling(A, "sea_ice")
lsm.partition(lam, -27.0, alpha=a)

# the conduction term on its own
lsm.lambda_g(6 * 3600.0, "sea_ice")     # 23.6 W m-2 K-1
lsm.lambda_g_steady("sea_ice_snow")     # 0.59
lsm.surface_admittance(3600.0, lsm.COLUMNS["sea_ice"])   # complex, with phase
lsm.damping_depth_m(86400.0)            # 0.17 m
lsm.effective_period_s(3.98)            # 8.8 days

# the emissivity convolution, and lambda_LW
lsm.planck_window_fraction(250.0)       # 0.234
lsm.broadband_emissivity(255.0, "sea_ice")
lsm.lambda_lw(255.0, "sea_ice")

# the whole table as a list of dicts, for a note or a slide
rows = lsm.coefficient_table(A, period_s=12 * 3600.0)
```

Or from the command line, which accepts every `turbulent_flux_response.py`
option plus this module's four:

```bash
python linearized_seb_model.py --region barrow --years 2022-2025 \
    --season-start 10-01 --season-end 03-31 --min-cloud-fraction 0.99 \
    --phase-mode fraction --period-hours 12 --column sea_ice --d-dlr -27 \
    --output-dir figures/linearized_seb
```

## Redraw everything

In [ ]:
lsm.fig_ground_admittance(out_dir=SAVE_DIR, dpi=DPI, A=A)
for fn in lsm.ALL_FIGURES:
    fn(A, out_dir=SAVE_DIR, dpi=DPI, period_s=PERIOD_S);